# Clase 3 — Evaluación de RAG

## ¿Cómo saber si un pipeline RAG es mejor que otro?

**Objetivos de la clase:**
- Distinguir entre métricas de **retrieval**, de **generación** y **específicas de RAG**.
- Construir un pequeño dataset de evaluación y usarlo para medir Recall@k y MRR.
- Usar un LLM como *juez* para evaluar respuestas de forma cualitativa.
- Calcular métricas de RAG (faithfulness, answer relevancy, context precision/recall) con `deepeval`.
- Comparar cuantitativamente el pipeline **naive** (Clase 1) vs el **avanzado** (Clase 2).

📎 Material de referencia: `Clase_8_RAG.pdf` (slide "Evaluación de RAG").

## 1. Tipos de métricas

| | Retrieval | Generación | Específicas de RAG |
|---|---|---|---|
| **Pregunta** | ¿Se recuperan los documentos relevantes? | ¿La respuesta es correcta, coherente y útil? | ¿La respuesta está fundamentada en el contexto y es relevante? |
| **Métricas** | Recall@k, Precision@k, MRR | Exact Match, F1, Accuracy, BLEU, ROUGE | faithfulness, answer relevancy, context precision, context recall |

Hoy vamos a calcular al menos un ejemplo de cada columna.

In [ ]:
# Instalar los paquetes necesarios
!pip install -q wikipedia python-dotenv==1.1.0 \
    langchain-community==0.3.25 langchain_openai==0.3.23 faiss-cpu==1.11.0 \
    rank_bm25==0.2.2 sentence-transformers deepeval==3.1.0 ipywidgets pandas matplotlib

In [ ]:
import os

try:
    from google.colab import userdata
    from google.colab.userdata import SecretNotFoundError
except ModuleNotFoundError:
    userdata = None
    SecretNotFoundError = type('SecretNotFoundError', (Exception,), {})


def obtener_secret(nombre, prompt_text=None):
    if userdata is not None:
        try:
            valor = userdata.get(nombre)
            if valor:
                return valor
        except SecretNotFoundError:
            # Secret not found in Colab userdata, proceed to environment variables
            pass

    valor = os.getenv(nombre)
    if valor:
        return valor

    return input(prompt_text or f"Ingresa {nombre}: ")

# OpenRouter se usa para el LLM (generación, LLM-as-judge, etc.) — crea tu key en openrouter.ai/keys
os.environ["OPENROUTER_API_KEY"] = obtener_secret("OPENROUTER_API_KEY", "OpenRouter API key: ")
# OpenAI se mantiene SOLO para los embeddings: OpenRouter no expone un endpoint de embeddings
os.environ["OPENAI_API_KEY"] = obtener_secret("OPENAI_API_KEY_DIPLOMADO", "OpenAI API key (para embeddings): ")

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "deepseek/deepseek-v4-flash"  # cámbialo por cualquier modelo de openrouter.ai/models

In [ ]:
from langchain_community.document_loaders import WikipediaLoader

WIKI_QUERY = "Cambio climático en Chile"

documento = WikipediaLoader(
    query=WIKI_QUERY, lang="es", load_max_docs=1, doc_content_chars_max=40000
).load()

print(f"Artículo cargado: '{documento[0].metadata.get('title')}' ({len(documento[0].page_content)} caracteres)")


## 2. Reconstruir los dos pipelines de las clases anteriores

Mismo código de las clases 1 y 2, condensado aquí para tener ambos pipelines disponibles en este notebook.

In [ ]:
import ipywidgets as widgets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from rank_bm25 import BM25Okapi

def dividir_en_chunks(documento, chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    chunks = splitter.split_documents(documento)
    for c in chunks:
        c.page_content = c.page_content.replace('\t', ' ')
    return chunks

def crear_llm(model=None, **kwargs):
    """ChatOpenAI apuntando a OpenRouter en vez de a la API de OpenAI directamente."""
    return ChatOpenAI(
        model=model or OPENROUTER_MODEL,
        base_url=OPENROUTER_BASE_URL,
        api_key=os.environ["OPENROUTER_API_KEY"],
        **kwargs,
    )

llm = crear_llm(temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

chunks = dividir_en_chunks(documento, chunk_size=1000, chunk_overlap=200)
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i

vectorstore = FAISS.from_documents(chunks, embeddings)

def crear_indice_bm25(chunks):
    tokenized = [c.page_content.split() for c in chunks]
    return BM25Okapi(tokenized)

bm25 = crear_indice_bm25(chunks)

def fusion_retrieval(vectorstore, bm25, chunks, query, k=5, alpha=0.5):
    epsilon = 1e-8
    n = len(chunks)
    bm25_scores = np.array(bm25.get_scores(query.split()))
    bm25_scores = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + epsilon)
    vector_results = vectorstore.similarity_search_with_score(query, k=n)
    vector_scores_por_id = np.zeros(n)
    for doc, score in vector_results:
        vector_scores_por_id[doc.metadata["chunk_id"]] = score
    vector_scores_por_id = 1 - (
        (vector_scores_por_id - vector_scores_por_id.min())
        / (vector_scores_por_id.max() - vector_scores_por_id.min() + epsilon)
    )
    combinado = alpha * vector_scores_por_id + (1 - alpha) * bm25_scores
    top_idx = np.argsort(combinado)[::-1][:k]
    return [chunks[i] for i in top_idx], combinado[top_idx]

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Responde ÚNICAMENTE con base en el contexto entregado. Si falta información dilo explícitamente."),
    ("human", "Contexto:\n{contexto}\n\nPregunta: {pregunta}"),
])
rag_chain = rag_prompt | llm | StrOutputParser()

def retriever_naive(pregunta, k):
    return vectorstore.as_retriever(search_kwargs={"k": k}).invoke(pregunta)

def retriever_avanzado(pregunta, k):
    docs, _ = fusion_retrieval(vectorstore, bm25, chunks, pregunta, k=k, alpha=0.5)
    return docs

def responder(retriever_fn, pregunta, k):
    docs = retriever_fn(pregunta, k)
    contexto = "\n\n".join(d.page_content for d in docs)
    return rag_chain.invoke({"contexto": contexto, "pregunta": pregunta}), docs

print("Pipelines de Clase 1 (naive) y Clase 2 (avanzado) listos.")


## 3. Dataset de evaluación

Para medir métricas necesitamos preguntas con una **respuesta esperada** y algunas **palabras clave** que deberían aparecer en un chunk relevante (una forma simplificada de marcar qué chunk es "correcto", sin etiquetar manualmente todo el corpus).

In [ ]:
eval_dataset = [
    {
        "question": "¿Qué es la megasequía y desde cuándo afecta a Chile central?",
        "expected_answer": "La megasequía es una secuencia ininterrumpida de años secos que afecta a Chile central desde 2010, con un déficit de precipitaciones de entre 20% y 40%.",
        "keywords": ["megasequía"],
    },
    {
        "question": "¿Cuánto han aumentado las emisiones de gases de efecto invernadero en Chile desde 1990?",
        "expected_answer": "Las emisiones de gases de efecto invernadero en Chile aumentaron 114,7% desde 1990.",
        "keywords": ["114,7", "114.7"],
    },
    {
        "question": "¿Qué establece la Ley Marco de Cambio Climático de Chile y cuándo se publicó?",
        "expected_answer": "La Ley Marco de Cambio Climático fue publicada en el Diario Oficial el 13 de junio de 2022 y establece la meta de carbono neutralidad para 2050.",
        "keywords": ["Ley Marco", "carbono neutralidad"],
    },
    {
        "question": "¿Cómo ha afectado el cambio climático a la biodiversidad en Chile?",
        "expected_answer": "El 71% de las especies de anfibios y el 83% de las especies de peces están en categoría de amenaza en Chile.",
        "keywords": ["anfibios"],
    },
    {
        "question": "¿Cuánto podría subir el nivel del mar en Chile hacia el año 2100?",
        "expected_answer": "Se proyecta un alza del nivel del mar de 20 cm entre los 30° y 60°S y de 25 cm entre los 20° y 30°S hacia el año 2100.",
        "keywords": ["nivel del mar"],
    },
    {
        "question": "¿Qué zonas de Chile han sido afectadas por aluviones relacionados al cambio climático?",
        "expected_answer": "El Norte Chico ha sido afectado por aluviones desde 2010, incluyendo el aluvión de Villa Santa Lucía en 2017.",
        "keywords": ["Villa Santa Lucía", "Norte Chico"],
    },
    {
        "question": "¿Qué institución lidera la política de cambio climático en Chile?",
        "expected_answer": "El Ministerio del Medio Ambiente, a través de su División de Cambio Climático, lidera la política climática en Chile.",
        "keywords": ["División de Cambio Climático"],
    },
    {
        "question": "¿Cuánto financiamiento recibió Chile del Fondo Verde del Clima entre 2016 y 2024?",
        "expected_answer": "Chile recibió 162.607.552 USD exclusivos del Fondo Verde del Clima entre 2016 y 2024.",
        "keywords": ["Fondo Verde del Clima"],
    },
]
print(f"{len(eval_dataset)} preguntas de evaluación.")

# Nota: antes las keywords incluían términos genéricos ("1990", "cm", "USD", "peces", "amenaza",
# "Ministerio del Medio Ambiente"). Con matching OR, bastaba con que apareciera CUALQUIERA para
# marcar el chunk como "relevante" — y esos términos aparecen en decenas de chunks del artículo,
# lo que saturaba Recall@k y MRR incluso a k bajos. Ahora cada keyword es lo bastante específica
# como para aparecer casi solo en el chunk que realmente responde la pregunta.

## 4. LLM-as-judge interactivo

Un juez LLM recibe pregunta + contexto + respuesta, y la califica en varias dimensiones. Es más flexible que las métricas de retrieval, pero también más subjetivo (depende del modelo juez y del prompt).

In [ ]:
import json, re
from langchain_core.prompts import PromptTemplate

def _json_desde_texto(texto):
    try:
        return json.loads(texto)
    except Exception:
        m = re.search(r"\{.*\}", texto, flags=re.DOTALL)
        if m:
            return json.loads(m.group(0))
        raise ValueError(f"No se pudo parsear JSON: {texto[:200]}")

juez_prompt = PromptTemplate.from_template(
    """Eres un evaluador estricto de sistemas RAG. Dada la pregunta, el contexto recuperado y la
respuesta generada, califica en escala 1-5:
- relevance: ¿la respuesta responde la pregunta?
- faithfulness: ¿la respuesta se basa solo en el contexto, sin inventar información?
- completeness: ¿la respuesta cubre la información relevante del contexto?

Devuelve SOLO un JSON compacto con estas claves exactas en minúscula y valores enteros. Sin texto adicional.

Pregunta: {pregunta}
Contexto: {contexto}
Respuesta: {respuesta}

JSON:"""
)
juez_chain = juez_prompt | llm | StrOutputParser()

def juez_llm(pregunta, respuesta, contexto):
    raw = juez_chain.invoke({"pregunta": pregunta, "contexto": contexto, "respuesta": respuesta})
    return _json_desde_texto(raw)

In [ ]:
@widgets.interact_manual(
    pregunta="¿Qué es la megasequía y desde cuándo afecta a Chile central?",
    respuesta="La megasequía es una secuencia ininterrumpida de años secos que afecta a Chile central desde 2010.",
    contexto="Chile central enfrenta una megasequía: una secuencia ininterrumpida de años secos desde 2010, con un déficit medio de precipitaciones de entre 20% y 40%...",
)
def probar_juez(pregunta, respuesta, contexto):
    resultado = juez_llm(pregunta, respuesta, contexto)
    display(pd.DataFrame([resultado]))

## 5. Métricas de retrieval: Recall@k, Precision@k y MRR

- **Recall@k**: ¿al menos uno de los `k` chunks recuperados es relevante?
- **Precision@k**: ¿qué fracción de los `k` chunks recuperados es relevante? (más sensible que Recall — no se satura tan rápido cuando k crece).
- **MRR (Mean Reciprocal Rank)**: promedio de `1 / posición_del_primer_chunk_relevante` — premia encontrar lo relevante *arriba* en el ranking, no solo en algún lugar.

Usamos la presencia de `keywords` en el chunk como proxy simplificado de "relevante".

⚠️ **Ojo con k grande**: si `k` se acerca al número total de chunks del corpus, casi cualquier método de retrieval termina devolviendo prácticamente todo — y ahí Recall@k y MRR convergen al mismo valor para naive y avanzado, no porque sean iguales, sino porque a esa escala la métrica ya no discrimina. Por eso el slider de abajo también muestra cuántos chunks tiene el corpus, para que compares k contra ese total.

In [ ]:
def es_relevante(chunk, keywords):
    texto = chunk.page_content.lower()
    return any(kw.lower() in texto for kw in keywords)

def recall_at_k(retriever_fn, dataset, k):
    aciertos = 0
    for item in dataset:
        docs = retriever_fn(item["question"], k)
        if any(es_relevante(d, item["keywords"]) for d in docs):
            aciertos += 1
    return aciertos / len(dataset)

def precision_at_k(retriever_fn, dataset, k):
    total = 0.0
    for item in dataset:
        docs = retriever_fn(item["question"], k)
        relevantes = sum(es_relevante(d, item["keywords"]) for d in docs)
        total += relevantes / k
    return total / len(dataset)

def mrr(retriever_fn, dataset, k):
    total = 0.0
    for item in dataset:
        docs = retriever_fn(item["question"], k)
        rr = 0.0
        for i, d in enumerate(docs, start=1):
            if es_relevante(d, item["keywords"]):
                rr = 1.0 / i
                break
        total += rr
    return total / len(dataset)

In [ ]:
@widgets.interact(k=widgets.IntSlider(value=3, min=1, max=15))
def comparar_retrieval(k):
    print(f"Corpus: {len(chunks)} chunks totales — k={k} es el {100 * k / len(chunks):.0f}% del corpus.")
    df = pd.DataFrame({
        "pipeline": ["naive", "avanzado"],
        f"recall@{k}": [recall_at_k(retriever_naive, eval_dataset, k), recall_at_k(retriever_avanzado, eval_dataset, k)],
        f"precision@{k}": [precision_at_k(retriever_naive, eval_dataset, k), precision_at_k(retriever_avanzado, eval_dataset, k)],
        "MRR": [mrr(retriever_naive, eval_dataset, k), mrr(retriever_avanzado, eval_dataset, k)],
    })
    display(df)
    df.set_index("pipeline").plot.bar(figsize=(6, 4), title=f"Retrieval @ k={k}")
    plt.ylim(0, 1.05)
    plt.show()

## 6. Métricas de generación con DeepEval

`deepeval` implementa las métricas específicas de RAG que vimos en la slide: **faithfulness**, **answer relevancy**, **contextual precision** y **contextual recall**. Todas se calculan usando un LLM auxiliar como juez — aquí, el mismo modelo de `OPENROUTER_MODEL` vía OpenRouter, envuelto en un adaptador para que DeepEval pueda usarlo.

In [ ]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric, ContextualPrecisionMetric, ContextualRecallMetric
from deepeval.test_case import LLMTestCase
from deepeval.models.base_model import DeepEvalBaseLLM

class ModeloJuezOpenRouter(DeepEvalBaseLLM):
    """Adapta nuestro ChatOpenAI (apuntando a OpenRouter) para que DeepEval lo use como juez."""
    def __init__(self, chat_model, nombre_modelo):
        self.chat_model = chat_model
        self.nombre_modelo = nombre_modelo

    def load_model(self):
        return self.chat_model

    def generate(self, prompt: str) -> str:
        return self.load_model().invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        respuesta = await self.load_model().ainvoke(prompt)
        return respuesta.content

    def get_model_name(self):
        return self.nombre_modelo

modelo_juez = ModeloJuezOpenRouter(crear_llm(temperature=0), OPENROUTER_MODEL)

metricas = [
    FaithfulnessMetric(model=modelo_juez),
    AnswerRelevancyMetric(model=modelo_juez),
    ContextualPrecisionMetric(model=modelo_juez),
    ContextualRecallMetric(model=modelo_juez),
]

def evaluar_pipeline(retriever_fn, nombre, k=5):
    filas = []
    for item in eval_dataset:
        respuesta, docs = responder(retriever_fn, item["question"], k)
        caso = LLMTestCase(
            input=item["question"],
            actual_output=respuesta,
            expected_output=item["expected_answer"],
            retrieval_context=[d.page_content for d in docs],
        )
        fila = {"pregunta": item["question"]}
        for metrica in metricas:
            metrica.measure(caso)
            fila[metrica.__class__.__name__] = round(metrica.score, 2)
        filas.append(fila)
    df = pd.DataFrame(filas)
    df["pipeline"] = nombre
    return df


In [ ]:
# Esta celda hace varias llamadas a la API por pipeline (una por métrica y pregunta) — puede tardar 1-2 minutos.
df_naive = evaluar_pipeline(retriever_naive, "naive", k=3)
df_avanzado = evaluar_pipeline(retriever_avanzado, "avanzado", k=5)
pd.concat([df_naive, df_avanzado], ignore_index=True)

In [ ]:
comparacion = pd.concat([df_naive, df_avanzado]).groupby("pipeline").mean(numeric_only=True)
comparacion.plot.bar(figsize=(8, 5), title="Comparación de métricas RAG: naive vs avanzado")
plt.ylabel("score promedio (0-1)")
plt.ylim(0, 1.05)
plt.show()
comparacion

## Ejercicios propuestos

1. **Ampliar el dataset de evaluación**
   - Agrega 2-3 preguntas nuevas a `eval_dataset`, con sus `keywords`. Vuelve a correr las secciones 5 y 6.

2. **Encontrar un caso de falla**
   - Busca una pregunta donde el pipeline avanzado tenga *peor* faithfulness o answer relevancy que el naive. ¿Por qué crees que pasa? (pista: el filtrado de relevancia puede descartar de más).

3. **De cualitativo a cuantitativo**
   - Toma una respuesta que el juez LLM (sección 4) calificó bajo en `faithfulness`. Revisa manualmente el contexto: ¿la respuesta realmente inventa información, o el juez se equivocó?

## 🔜 Próxima clase

Nuestros pipelines siguen fallando en un tipo de pregunta particular: las que requieren **conectar información entre varias entidades** (razonamiento multi-hop), algo que la similitud vectorial por sí sola no resuelve bien. En la **Clase 4 — GraphRAG** vamos a representar el conocimiento como un **grafo** y usarlo, junto con RAG vectorial, para responder este tipo de preguntas.